# Asset Types & Depositary Receipts

SET lists far more than common stocks. The same APIs serve **ETFs**, **Depositary Receipts (DRs)**,
**derivative warrants (DWs)**, warrants, preferred shares and unit trusts — all mixed together, told
apart only by a single-letter `securityType` code.

This notebook covers two things:

1. **Asset types** — classify any symbol (`AssetType`), and slice the whole universe by type.
2. **Depositary Receipts** — the issuer/underlying details behind a DR like `GOOG80`, the
   TradingView **"Indicative Price"** chart SET links to, and how to compute that indicative price
   (`underlying price × FX ÷ conversion ratio`) yourself.

Endpoints: `GET /api/set/stock/{symbol}/profile` · `GET /api/set/dr/{symbol}/profile` ·
`POST scanner.tradingview.com/global/scan`

## Setup

Install the library (uncomment if needed), then import the helpers.

In [1]:
# !pip install settfex

from settfex.services.set import (
    AssetType,
    Stock,
    get_dr_indicative_price,
    get_dr_profile,
    get_stock_list,
)

## 1. What kind of instrument is this?

`Stock.get_asset_type()` reads the symbol's profile once, caches it on the instance, and maps SET's
`securityType` code to a friendly `AssetType`. It never raises on an unknown code — a code SET adds
tomorrow simply comes back as `AssetType.UNKNOWN`.

In [2]:
symbols = ["CPALL", "1DIV", "GOOG80", "AAV01C2609T", "SCBSET"]

for symbol in symbols:
    asset_type = await Stock(symbol).get_asset_type()
    print(f"{symbol:<14} {asset_type}")

CPALL          stock
1DIV           etf
GOOG80         dr
AAV01C2609T    dw
SCBSET         unit_trust


`AssetType` is a `StrEnum`, so it prints as its bare value and compares equal to a plain string —
handy in f-strings, JSON, and `if` checks alike.

In [3]:
asset_type = await Stock("GOOG80").get_asset_type()

print(f"repr:            {asset_type!r}")
print(f"str:             {asset_type}")
print(f"== 'dr':         {asset_type == 'dr'}")
print(f"is DR:           {asset_type is AssetType.DEPOSITARY_RECEIPT}")

repr:            <AssetType.DEPOSITARY_RECEIPT: 'dr'>
str:             dr
== 'dr':         True
is DR:           True


## 2. Slice the whole universe by type

Every `StockSymbol` in the stock list exposes the same `asset_type` property, and
`filter_by_asset_type()` accepts an `AssetType`, its value (`"dr"`), or the raw SET code (`"X"`).

Fetching with `include_indices=False` skips the index-membership enrichment — we don't need it here,
and it makes the call noticeably faster.

In [4]:
from collections import Counter

stock_list = await get_stock_list(include_indices=False)
counts = Counter(s.asset_type for s in stock_list.security_symbols)

print(f"{stock_list.count:,} listed securities\n")
for asset_type, count in counts.most_common():
    print(f"  {asset_type:<26} {count:>5,}")

4,054 listed securities

  dw                         1,651
  stock                        930
  stock_foreign                864
  dr                           493
  warrant                       85
  etf                           13
  preferred_stock                8
  preferred_stock_foreign        8
  unit_trust                     2


In [5]:
drs = stock_list.filter_by_asset_type(AssetType.DEPOSITARY_RECEIPT)
etfs = stock_list.filter_by_asset_type("etf")  # by value
dws = stock_list.filter_by_asset_type("V")  # by raw SET securityType code

print(f"DRs:  {len(drs):>4}   e.g. {[s.symbol for s in drs[:6]]}")
print(f"ETFs: {len(etfs):>4}   e.g. {[s.symbol for s in etfs[:6]]}")
print(f"DWs:  {len(dws):>4}   e.g. {[s.symbol for s in dws[:4]]}")

DRs:   493   e.g. ['AAOI03', 'AAPL01', 'AAPL03', 'AAPL19', 'AAPL80', 'ABBV19']
ETFs:   13   e.g. ['1DIV', '1I01BSET50', '2I01BSET50', '2X01BSET50', 'ABFTH', 'BMSCG']
DWs:  1651   e.g. ['AAV01C2609T', 'AAV01C2703T', 'AAV13C2608A', 'AAV13C2608B']


This is the practical way to build a *tradable stock* universe — DWs alone outnumber ordinary
listed companies, so a naive "all symbols" screen is mostly derivatives.

In [6]:
tradable = [
    s
    for s in stock_list.security_symbols
    if s.asset_type in (AssetType.STOCK, AssetType.STOCK_FOREIGN)
]
print(f"{len(tradable):,} common stocks out of {stock_list.count:,} listed securities")

1,794 common stocks out of 4,054 listed securities


## 3. Inside a Depositary Receipt

A DR is a Thai-listed wrapper around a foreign security. `get_dr_profile()` returns who issued it,
what it tracks, on which exchange, and — crucially — the **conversion ratio**.

In [7]:
profile = await get_dr_profile("GOOG80")

print(f"Symbol:       {profile.symbol}")
print(f"Name:         {profile.name}")
print(f"Issuer:       {profile.issuer} — {profile.issuer_name}")
print(f"Underlying:   {profile.underlying} ({profile.underlying_name})")
print(f"Exchange:     {profile.underlying_exchange}")
print(f"Ratio:        {profile.conversion_ratio}")
print(f"Session:      {profile.trading_session}")
print(f"Fractional:   {profile.fractional_trade}")

Symbol:       GOOG80
Name:         Depositary Receipt on GOOG Issued by KTB
Issuer:       KTB — KRUNG THAI BANK PUBLIC COMPANY LIMITED
Underlying:   GOOG (ALPHABET INC. (GOOG))
Exchange:     The Nasdaq Global Select Market
Ratio:        2,000 : 1
Session:      Day & Night Session
Fractional:   False


### The "Indicative Price" link

On SET's DR pages there is an **Indicative Price** menu item pointing at a TradingView chart. The
profile carries both that URL and the expression it charts.

In [8]:
print(profile.tradingview_url)

expression = profile.indicative_expression
print(f"\nexpression: {expression.expression}")
print(f"tickers:    {expression.tickers}")
print(f"ratio:      {expression.ratio}")

https://th.tradingview.com/chart/?symbol=NASDAQ%3AGOOG*FX_IDC%3AUSDTHB%2F2000.0

expression: NASDAQ:GOOG*FX_IDC:USDTHB/2000.0
tickers:    ['NASDAQ:GOOG', 'FX_IDC:USDTHB']
ratio:      2000.0


Read the expression as arithmetic: **`NASDAQ:GOOG` × `FX_IDC:USDTHB` ÷ `2000`** — the underlying's
price in USD, converted to THB, divided by how many DR units represent one share.

> Some DRs return a `null` `indicativePriceSymbol` (HERMES80, BYDCOM80, NDX01 among them). The
> library recovers the expression from the URL's `symbol` query parameter, so
> `indicative_expression` still works.

In [9]:
hermes = await get_dr_profile("HERMES80")

print(f"indicative_price_symbol: {hermes.indicative_price_symbol}")
print(f"recovered from URL:      {hermes.indicative_expression.expression}")

indicative_price_symbol: None
recovered from URL:      EURONEXT:RMS*FX_IDC:EURTHB/10000.0


## 4. The indicative price

`get_dr_indicative_price()` fetches every leg of the expression from TradingView in **one** batch
request and evaluates it, returning the fair value in THB plus each leg's quote.

In [10]:
price = await get_dr_indicative_price("GOOG80")

for leg in price.legs:
    print(f"  {leg.ticker:<16} {leg.close:>10,.4f} {leg.currency}   ({leg.update_mode})")

print(f"\n  {'÷ ratio':<16} {price.ratio:>10,.1f}")
print(f"  {'= indicative':<16} {price.indicative_price:>10,.4f} THB")
print(f"\nas of {price.as_of:%Y-%m-%d %H:%M:%S %Z} · delayed={price.is_delayed}")

  NASDAQ:GOOG        356.6500 USD   (delayed_streaming_900)
  FX_IDC:USDTHB       33.3300 THB   (streaming)

  ÷ ratio             2,000.0
  = indicative         5.9436 THB

as of 2026-08-03 11:55:27 +07 · delayed=True


The `underlying` and `fx` helpers pick out the legs, so you can show the arithmetic explicitly.

In [11]:
underlying, fx = price.underlying, price.fx

print(
    f"{underlying.close:,.2f} {underlying.currency} "
    f"× {fx.close:,.4f} THB/{underlying.currency} "
    f"÷ {price.ratio:,.0f} "
    f"= {price.indicative_price:,.4f} THB"
)

356.65 USD × 33.3300 THB/USD ÷ 2,000 = 5.9436 THB


## 5. `get_latest_price()` is DR-aware

For a DR, "the last price" is ambiguous: the DR's own last trade on SET, or what it is *worth* right
now given where the underlying is trading? `Stock.get_latest_price()` answers with the **indicative**
price for DRs, because the underlying keeps moving long after SET closes.

The result is a `DrIndicativeQuotation` — a `Quotation` subclass whose `volume`/`change` are `None`
(it is a fair value, not a trade) and whose `.indicative` field carries the full computation.

In [12]:
dr = Stock("GOOG80")

indicative = await dr.get_latest_price()
traded = await dr.get_latest_price(prefer_dr_indicative=False)

print(f"indicative : {indicative.price:>8,.4f} THB   ({type(indicative).__name__})")
print(f"SET traded : {traded.price:>8,.4f} THB   ({type(traded).__name__})")
print(f"gap        : {indicative.price - traded.price:>+8,.4f} THB")

indicative :   5.9436 THB   (DrIndicativeQuotation)
SET traded :   6.0000 THB   (Quotation)
gap        :  -0.0564 THB


A gap between the two is **normal**, not an error: the exchange leg is ~15 minutes delayed, and US
or European hours barely overlap SET's. That spread is exactly what a DR arbitrage screen looks for.

Non-DR symbols are unaffected — they go straight to SET chart data as before.

In [13]:
quote = await Stock("CPALL").get_latest_price()

print(f"CPALL: {quote.price} THB at {quote.local_datetime} ({type(quote).__name__})")
print(f"volume: {quote.volume:,.0f}   <- a real SET trade, so volume is populated")

CPALL: 48.25 THB at 2026-08-03 11:54:00 (Quotation)
volume: 8,800   <- a real SET trade, so volume is populated


## Summary

- `Stock.get_asset_type()` → `AssetType` (`stock`, `stock_foreign`, `preferred_stock`,
  `preferred_stock_foreign`, `warrant`, `dw`, `etf`, `unit_trust`, `dr`, `unknown`); cached per
  instance, never raises on a new code.
- `StockSymbol.asset_type` and `stock_list.filter_by_asset_type(...)` slice the whole universe —
  accepting an enum, its value, or the raw SET `securityType` code.
- `get_dr_profile(symbol)` → issuer, underlying, exchange, conversion ratio, and the TradingView
  `tradingview_url`; `indicative_expression` parses the pricing formula (recovering it from the URL
  when the symbol field is null).
- `get_dr_indicative_price(symbol)` → THB fair value = product of leg prices ÷ ratio, with each
  leg's quote and delay status.
- `Stock.get_latest_price()` returns the indicative price for DRs (opt out with
  `prefer_dr_indicative=False`); on any TradingView failure it falls back to SET chart data.

Bonds are **not** covered by these APIs at all — there is deliberately no `AssetType.BOND`.

See `docs/settfex/services/set/profile_dr.md` and
`docs/settfex/services/set/dr_indicative_price.md` for the full reference.